# Droplet Tracking Pipeline using YOLO and Hungarian Algorithm

This notebook provides a complete end-to-end solution for droplet tracking, trajectory analysis, and kinematic computation using YOLO object detection and Hungarian algorithm for optimal data association.

## Key Improvements:
- **Consistent Track IDs**: Deterministic HSV-based color mapping for consistent droplet identification
- **Incremental Path Drawing**: Paths grow as droplets move (sliding window of last 30 frames)
- **Thin Paths**: Reduced line thickness (1px) for cleaner visualization
- **Hungarian Algorithm**: Optimal assignment of detections to tracks

In [ ]:
# ============================================================================
# SECTION 1: Load YOLO Model and Frame Data
# ============================================================================
# Import required libraries
from ultralytics import YOLO
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from scipy import stats
import glob
import os
from colorsys import hsv_to_rgb

print("Libraries imported successfully")

# Configuration
MODEL_PATH = "runs/detect/train3/weights/best.pt"  # Path to trained YOLO model
FRAME_FOLDER = "videos"  # Folder containing frame images
OUTPUT_FOLDER = "output_droplet_tracking"
CSV_OUTPUT = "droplet_trajectories_final.csv"
VIDEO_OUTPUT = "droplet_tracking_output.mp4"

# Create output folder
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Load YOLO model
print("Loading YOLO model...")
model = YOLO(MODEL_PATH)
print(f"Model loaded from: {MODEL_PATH}")

# Load frame paths
frame_paths = sorted(glob.glob(os.path.join(FRAME_FOLDER, "*.png")))
if not frame_paths:
    print(f"Warning: No PNG files found in {FRAME_FOLDER}")
    # Fallback: try jpg files
    frame_paths = sorted(glob.glob(os.path.join(FRAME_FOLDER, "*.jpg")))

if frame_paths:
    sample_frame = cv2.imread(frame_paths[0])
    if sample_frame is not None:
        h, w = sample_frame.shape[:2]
        print(f"Found {len(frame_paths)} frames")
        print(f"Frame dimensions: {w}x{h}")
        
        # Display sample frame
        plt.figure(figsize=(8, 6))
        plt.imshow(cv2.cvtColor(sample_frame, cv2.COLOR_BGR2RGB))
        plt.title("Sample Frame")
        plt.axis("off")
        plt.tight_layout()
        plt.show()
else:
    print("No frames found!")


In [ ]:
# ============================================================================
# SECTION 2: Implement Hungarian Algorithm Tracker
# ============================================================================

class KalmanTracker:
    """Simple Kalman filter-based tracker for droplet tracking."""
    
    def __init__(self, x, y, track_id):
        self.id = track_id
        # State: [x, y, vx, vy]
        self.state = np.array([x, y, 0.0, 0.0], dtype=float)
        
        # Covariance matrix
        self.P = np.eye(4) * 100.0
        
        # State transition matrix
        self.F = np.array([
            [1, 0, 1, 0],
            [0, 1, 0, 1],
            [0, 0, 1, 0],
            [0, 0, 0, 1]
        ], dtype=float)
        
        # Measurement matrix (we measure x, y only)
        self.H = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0]
        ], dtype=float)
        
        # Measurement noise
        self.R = np.eye(2) * 5.0
        
        # Process noise
        self.Q = np.eye(4) * 1.0
        
        self.missed_frames = 0
        self.age = 0
    
    def predict(self):
        """Predict next position using motion model."""
        self.state = self.F @ self.state
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.state[:2].copy()
    
    def update(self, measurement):
        """Update track with new measurement (Kalman update)."""
        z = np.array(measurement, dtype=float)
        
        # Innovation
        y = z - self.H @ self.state
        
        # Innovation covariance
        S = self.H @ self.P @ self.H.T + self.R
        
        # Kalman gain
        K = self.P @ self.H.T @ np.linalg.inv(S)
        
        # Update state
        self.state = self.state + K @ y
        
        # Update covariance
        self.P = (np.eye(4) - K @ self.H) @ self.P
        
        self.missed_frames = 0
        self.age += 1
    
    def get_position(self):
        """Get current position."""
        return self.state[:2].copy()
    
    def get_velocity(self):
        """Get current velocity."""
        return self.state[2:4].copy()


class TrackerManager:
    """Manages multiple tracks using Hungarian algorithm for assignment."""
    
    def __init__(self, max_distance=50, max_missed_frames=30, min_track_length=5):
        self.trackers = []
        self.next_id = 0
        self.max_distance = max_distance
        self.max_missed_frames = max_missed_frames
        self.min_track_length = min_track_length
    
    def update(self, detections):
        """
        Update tracker with new detections using Hungarian algorithm.
        
        Args:
            detections: numpy array of shape (N, 2) with [x, y] coordinates
        """
        detections = np.array(detections, dtype=float)
        
        # Predict positions for all current trackers
        predictions = []
        for trk in self.trackers:
            predictions.append(trk.predict())
        
        predictions = np.array(predictions) if predictions else np.empty((0, 2))
        
        # Handle initial case (no active trackers)
        if len(predictions) == 0:
            for det in detections:
                self.trackers.append(KalmanTracker(det[0], det[1], self.next_id))
                self.next_id += 1
            return
        
        # Compute cost matrix (Euclidean distances)
        cost_matrix = np.zeros((len(self.trackers), len(detections)))
        for i, pred in enumerate(predictions):
            for j, det in enumerate(detections):
                cost_matrix[i, j] = np.linalg.norm(pred - det)
        
        # Hungarian algorithm (linear sum assignment)
        row_indices, col_indices = linear_sum_assignment(cost_matrix)
        
        assigned_trackers = set()
        assigned_detections = set()
        
        # Update matched pairs
        for r, c in zip(row_indices, col_indices):
            if cost_matrix[r, c] < self.max_distance:
                self.trackers[r].update(detections[c])
                assigned_trackers.add(r)
                assigned_detections.add(c)
        
        # Mark unmatched trackers as missed
        for i, trk in enumerate(self.trackers):
            if i not in assigned_trackers:
                trk.missed_frames += 1
        
        # Remove stale tracks
        self.trackers = [t for t in self.trackers if t.missed_frames < self.max_missed_frames]
        
        # Create new trackers for unmatched detections
        for j, det in enumerate(detections):
            if j not in assigned_detections:
                self.trackers.append(KalmanTracker(det[0], det[1], self.next_id))
                self.next_id += 1
    
    def get_active_trackers(self):
        """Return all active trackers."""
        return self.trackers.copy()


print("Tracker classes defined successfully")


In [ ]:
# ============================================================================
# SECTION 3: Run Detection and Tracking Pipeline
# ============================================================================

# Parameters for detection and tracking
YOLO_CONF = 0.5
MIN_BOX_SIZE = 10
MAX_BOX_SIZE = 80
MAX_DISTANCE = 30
MAX_MISSED_FRAMES = 40

# Initialize tracker manager
tracker = TrackerManager(
    max_distance=MAX_DISTANCE,
    max_missed_frames=MAX_MISSED_FRAMES
)

# Storage for tracking history
frame_data = []
trajectory_history = {}

print(f"Processing {len(frame_paths)} frames...")
print(f"YOLO confidence: {YOLO_CONF}, Box size range: {MIN_BOX_SIZE}-{MAX_BOX_SIZE}")

# Process each frame
for frame_idx, img_path in enumerate(frame_paths):
    frame = cv2.imread(img_path)
    if frame is None:
        print(f"Warning: Could not load frame {frame_idx}")
        continue
    
    # YOLO inference
    results = model(frame, conf=YOLO_CONF, verbose=False)
    
    # Extract detections (bounding box centers)
    detections = []
    if len(results[0].boxes) > 0:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        
        for box in boxes:
            x1, y1, x2, y2 = box
            box_width = x2 - x1
            box_height = y2 - y1
            
            # Filter by size constraints
            if MIN_BOX_SIZE < box_width < MAX_BOX_SIZE and MIN_BOX_SIZE < box_height < MAX_BOX_SIZE:
                x_center = (x1 + x2) / 2.0
                y_center = (y1 + y2) / 2.0
                detections.append([x_center, y_center])
    
    detections = np.array(detections) if detections else np.empty((0, 2))
    
    # Update tracker with Hungarian algorithm
    tracker.update(detections)
    
    # Record track data
    active_trackers = tracker.get_active_trackers()
    for trk in active_trackers:
        pos = trk.get_position()
        vel = trk.get_velocity()
        
        frame_data.append({
            'frame': frame_idx,
            'track_id': trk.id,
            'x': pos[0],
            'y': pos[1],
            'vx': vel[0],
            'vy': vel[1],
            'speed': np.linalg.norm(vel)
        })
        
        # Keep trajectory history
        if trk.id not in trajectory_history:
            trajectory_history[trk.id] = []
        trajectory_history[trk.id].append(tuple(pos))
    
    if (frame_idx + 1) % max(1, len(frame_paths) // 10) == 0:
        print(f"  Processed {frame_idx + 1}/{len(frame_paths)} frames, {len(active_trackers)} active tracks")

print(f"\nTracking complete: {len(frame_data)} detections, {len(trajectory_history)} unique tracks")

# Create DataFrame from tracking data
tracks_df = pd.DataFrame(frame_data)
print(f"\nTracking data summary:")
print(tracks_df.head(10))


In [ ]:
# ============================================================================
# SECTION 4: Visualize Tracked Trajectories (FIXED)
# ============================================================================

# Helper function for CONSISTENT colors
def get_track_color(track_id):
    """Generate a consistent, deterministic color for each track ID."""
    # Use hash to create consistent colors independent of random state
    hash_val = hash(track_id) % 256
    # Create distinct colors using HSV color space and convert to BGR
    hue = (hash_val % 256) / 256.0
    sat = 0.8
    val = 0.9
    r, g, b = hsv_to_rgb(hue, sat, val)
    return (int(b * 255), int(g * 255), int(r * 255))  # BGR format

print(f"Track color function created - consistent IDs using HSV hash")

# Setup video writer with incremental path drawing
if frame_paths:
    sample = cv2.imread(frame_paths[0])
    frame_h, frame_w = sample.shape[:2]
    
    output_video_path = os.path.join(OUTPUT_FOLDER, VIDEO_OUTPUT)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_video_path, fourcc, 10.0, (frame_w, frame_h))
    
    print(f"Creating visualization video: {output_video_path}")
    print(f"Video dimensions: {frame_w}x{frame_h}")
    print(f"✓ Path drawing: INCREMENTAL (sliding window of last 30 frames)")
    print(f"✓ Path thickness: 1 pixel (reduced from 2)")
    print(f"✓ Track ID colors: Deterministic & consistent")
    
    # Track history per frame (incremental drawing)
    incremental_history = {}
    PATH_LENGTH_WINDOW = 30  # Only show last N frames of trajectory
    
    # Process frames for visualization
    for frame_idx, img_path in enumerate(frame_paths):
        frame = cv2.imread(img_path)
        if frame is None:
            continue
        
        # Get current frame detections
        frame_tracks = tracks_df[tracks_df['frame'] == frame_idx]
        
        for _, track in frame_tracks.iterrows():
            track_id = int(track['track_id'])
            x, y = int(track['x']), int(track['y'])
            color = get_track_color(track_id)
            
            # Update incremental history for this track
            if track_id not in incremental_history:
                incremental_history[track_id] = []
            incremental_history[track_id].append((x, y))
            
            # Keep only recent history (sliding window) - INCREMENTAL DRAWING
            if len(incremental_history[track_id]) > PATH_LENGTH_WINDOW:
                incremental_history[track_id].pop(0)
            
            # Draw polyline with REDUCED thickness (1 instead of 2) - THIN PATHS
            if len(incremental_history[track_id]) > 1:
                pts = np.array(incremental_history[track_id], dtype=np.int32)
                cv2.polylines(frame, [pts], False, color, 1)  # Thickness = 1
            
            # Draw circle at current position
            cv2.circle(frame, (x, y), 8, color, 2)
            cv2.circle(frame, (x, y), 3, (255, 255, 255), -1)
            
            # Draw track ID label (CONSISTENT ID now)
            cv2.putText(
                frame,
                f"id:{track_id}",
                (x + 10, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )
        
        writer.write(frame)
        
        if (frame_idx + 1) % max(1, len(frame_paths) // 10) == 0:
            print(f"  Processed {frame_idx + 1}/{len(frame_paths)} frames for visualization")
    
    writer.release()
    print(f"\n✓ Video saved: {output_video_path}")
    
    # Display a sample annotated frame
    if len(frame_paths) > 0:
        sample_idx = len(frame_paths) // 2
        sample_frame = cv2.imread(frame_paths[sample_idx])
        
        # Draw tracking for sample frame with incremental history
        current_tracks = tracks_df[tracks_df['frame'] == sample_idx]
        for _, track in current_tracks.iterrows():
            track_id = int(track['track_id'])
            x, y = int(track['x']), int(track['y'])
            color = get_track_color(track_id)
            
            # Draw recent trajectory for this track (incremental)
            track_history = incremental_history.get(track_id, [])
            if len(track_history) > 1:
                pts = np.array(track_history, dtype=np.int32)
                cv2.polylines(sample_frame, [pts], False, color, 1)  # Thin line
            
            # Draw position
            cv2.circle(sample_frame, (x, y), 8, color, 2)
            cv2.circle(sample_frame, (x, y), 3, (255, 255, 255), -1)
            cv2.putText(
                sample_frame,
                f"id:{track_id}",
                (x + 10, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )
        
        plt.figure(figsize=(10, 8))
        plt.imshow(cv2.cvtColor(sample_frame, cv2.COLOR_BGR2RGB))
        plt.title(f"Sample Annotated Frame ({sample_idx}) - Incremental Paths (Last 30 Frames)")
        plt.axis("off")
        plt.tight_layout()
        plt.show()


In [ ]:
# ============================================================================
# SECTION 5: Compute Trajectory Statistics and Kinematics
# ============================================================================

print("Computing trajectory statistics and kinematics...\n")

# Compute additional kinematic quantities
tracks_df['speed'] = np.sqrt(tracks_df['vx']**2 + tracks_df['vy']**2)

# Compute acceleration (frame-to-frame change in velocity)
tracks_df = tracks_df.sort_values(['track_id', 'frame']).reset_index(drop=True)
tracks_df['ax'] = tracks_df.groupby('track_id')['vx'].diff().fillna(0)
tracks_df['ay'] = tracks_df.groupby('track_id')['vy'].diff().fillna(0)
tracks_df['acceleration'] = np.sqrt(tracks_df['ax']**2 + tracks_df['ay']**2)

# Compute ensemble mean square displacement (MSD)
print("Computing Mean Square Displacement...")
msd_list = []
max_tau = min(100, len(tracks_df) // 10)

for tau in range(1, max_tau):
    displacements = []
    
    for track_id in tracks_df['track_id'].unique():
        track_data = tracks_df[tracks_df['track_id'] == track_id].reset_index(drop=True)
        
        if len(track_data) > tau:
            dx = track_data['x'].iloc[tau:].values - track_data['x'].iloc[:-tau].values
            dy = track_data['y'].iloc[tau:].values - track_data['y'].iloc[:-tau].values
            r_squared = dx**2 + dy**2
            displacements.extend(r_squared)
    
    if displacements:
        msd = np.mean(displacements)
        msd_list.append(msd)

# Compute pair-wise distances
print("Computing pair-wise distances...")
pair_distances = []

for frame_idx in tracks_df['frame'].unique():
    frame_tracks = tracks_df[tracks_df['frame'] == frame_idx]
    positions = frame_tracks[['x', 'y']].values
    
    if len(positions) > 1:
        for i in range(len(positions)):
            for j in range(i + 1, len(positions)):
                distance = np.linalg.norm(positions[i] - positions[j])
                pair_distances.append(distance)

print(f"Computed {len(pair_distances)} pair-wise distances\n")

# Summary statistics
print("=" * 60)
print("KINEMATIC SUMMARY STATISTICS")
print("=" * 60)
print(f"Total frames processed: {len(frame_paths)}")
print(f"Total unique tracks: {len(tracks_df['track_id'].unique())}")
print(f"Total detections: {len(tracks_df)}")
print(f"\nSpeed statistics:")
print(f"  Mean speed: {tracks_df['speed'].mean():.3f} pixels/frame")
print(f"  Std speed: {tracks_df['speed'].std():.3f} pixels/frame")
print(f"  Min speed: {tracks_df['speed'].min():.3f} pixels/frame")
print(f"  Max speed: {tracks_df['speed'].max():.3f} pixels/frame")
print(f"\nAcceleration statistics:")
print(f"  Mean acceleration: {tracks_df['acceleration'].mean():.3f} pixels/frame²")
print(f"  Std acceleration: {tracks_df['acceleration'].std():.3f} pixels/frame²")
print(f"\nPair distance statistics:")
print(f"  Mean: {np.mean(pair_distances):.3f} pixels")
print(f"  Std: {np.std(pair_distances):.3f} pixels")
print(f"  Min: {np.min(pair_distances):.3f} pixels")
print(f"  Max: {np.max(pair_distances):.3f} pixels")
print("=" * 60)

# Plot speed distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.hist(tracks_df['speed'].dropna(), bins=40, color='steelblue', edgecolor='black')
plt.xlabel('Speed (pixels/frame)')
plt.ylabel('Frequency')
plt.title('Droplet Speed Distribution')
plt.grid(alpha=0.3)

# Plot acceleration distribution
plt.subplot(1, 3, 2)
plt.hist(tracks_df['acceleration'].dropna(), bins=40, color='coral', edgecolor='black')
plt.xlabel('Acceleration (pixels/frame²)')
plt.ylabel('Frequency')
plt.title('Droplet Acceleration Distribution')
plt.grid(alpha=0.3)

# Plot pair distance distribution
plt.subplot(1, 3, 3)
plt.hist(pair_distances, bins=40, color='lightgreen', edgecolor='black')
plt.xlabel('Distance (pixels)')
plt.ylabel('Frequency')
plt.title('Pair Distance Distribution')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Plot velocity field
print("\nPlotting velocity field...")
plt.figure(figsize=(10, 8))
plt.quiver(tracks_df['x'], tracks_df['y'], tracks_df['vx'], tracks_df['vy'])
plt.xlabel('X (pixels)')
plt.ylabel('Y (pixels)')
plt.title('Droplet Velocity Field')
plt.gca().set_aspect('equal')
plt.tight_layout()
plt.show()

# Plot MSD
if msd_list:
    print("Plotting Mean Square Displacement...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(msd_list) + 1), msd_list, 'o-', linewidth=2, markersize=4)
    plt.xlabel('Time lag τ (frames)')
    plt.ylabel('MSD (pixels²)')
    plt.title('Ensemble Mean Square Displacement')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================================
# SECTION 6: Export Results to CSV and Video
# ============================================================================

print("Exporting results...\n")

# Save tracking data to CSV
csv_path = os.path.join(OUTPUT_FOLDER, CSV_OUTPUT)
tracks_df.to_csv(csv_path, index=False)
print(f"Saved tracking data to: {csv_path}")
print(f"CSV columns: {list(tracks_df.columns)}")
print(f"CSV shape: {tracks_df.shape}")

# Display first few rows
print("\nFirst 10 rows of tracking data:")
print(tracks_df.head(10))

# Summary statistics
print("\n" + "=" * 70)
print("FINAL SUMMARY REPORT")
print("=" * 70)

num_frames = len(frame_paths)
num_tracks = len(tracks_df['track_id'].unique())
num_detections = len(tracks_df)
avg_track_length = num_detections / num_tracks if num_tracks > 0 else 0

print(f"\nProcessing Results:")
print(f"  ✓ Total frames processed: {num_frames}")
print(f"  ✓ Unique droplet tracks: {num_tracks}")
print(f"  ✓ Total detections: {num_detections}")
print(f"  ✓ Average track length: {avg_track_length:.1f} frames")

print(f"\nOutput Files:")
print(f"  ✓ Tracking CSV: {csv_path}")
print(f"  ✓ Visualization video: {os.path.join(OUTPUT_FOLDER, VIDEO_OUTPUT)}")

print(f"\nTracking Parameters:")
print(f"  YOLO confidence threshold: {YOLO_CONF}")
print(f"  Detection size range: {MIN_BOX_SIZE}-{MAX_BOX_SIZE} pixels")
print(f"  Max association distance: {MAX_DISTANCE} pixels")
print(f"  Max frames to keep lost tracks: {MAX_MISSED_FRAMES}")

print(f"\nKey Statistics:")
print(f"  Speed range: {tracks_df['speed'].min():.3f} - {tracks_df['speed'].max():.3f} pixels/frame")
print(f"  Mean speed: {tracks_df['speed'].mean():.3f} ± {tracks_df['speed'].std():.3f} pixels/frame")
print(f"  Acceleration range: {tracks_df['acceleration'].min():.3f} - {tracks_df['acceleration'].max():.3f} px/frame²")
print(f"  Mean acceleration: {tracks_df['acceleration'].mean():.3f} ± {tracks_df['acceleration'].std():.3f} px/frame²")

print(f"\nVisualization Improvements:")
print(f"  ✓ Track ID colors: Deterministic (consistent across runs)")
print(f"  ✓ Path drawing: Incremental (sliding window of 30 frames)")
print(f"  ✓ Path thickness: 1 pixel (reduced from 2)")

print("\n" + "=" * 70)
print("Analysis Complete!")
print("=" * 70)
